<a href="https://colab.research.google.com/github/sscoconut64/Joey_DTSC3020_Fall2025/blob/main/Assignment_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Data Quality Check

1.1 Using Python (pandas, matplotlib, or seaborn), load and inspect the Assignment 2 dataset.

In [2]:
# Write your code here

from google.colab import files
uploaded = files.upload()


Saving Assignment 2 dataset.csv to Assignment 2 dataset.csv


Write code to explore the data distribution (e.g., region, type, year) and check whether there is any bias. Provide both the code and your interpretation.

In [3]:
# Write your code here
import pandas as pd
import io

if 'Assignment 2 dataset.csv' in uploaded:
    df = pd.read_csv(io.BytesIO(uploaded['Assignment 2 dataset.csv']))

    if 'type' in df.columns:
        print("\nCounts of 'organic' versus 'conventional' in 'type' column:")
        type_counts = df['type'].value_counts()
        print(type_counts)

    if 'year' in df.columns:
        print("\nCounts of each year in 'year' column:")
        year_counts = df['year'].value_counts()
        print(year_counts)

    if 'region' in df.columns:
        print("\nCounts of each region in 'region' column:")
        region_counts = df['region'].value_counts()
        print(region_counts)

# My impression upon looking at the data is that the 'type' and 'region' columns
# are very balanced. The 'year' column is also mostly balanced accept for a
# relatively large lack of data for the year 2018 (and also the bizzare
# 1 piece of data for 1904)


Counts of 'organic' versus 'conventional' in 'type' column:
type
organic         9127
conventional    9126
Name: count, dtype: int64

Counts of each year in 'year' column:
year
2017    5722
2016    5616
2015    5615
2018    1300
1904       1
Name: count, dtype: int64

Counts of each region in 'region' column:
region
WestTexNewMexico       340
Albany                 338
BaltimoreWashington    338
Boise                  338
Boston                 338
Atlanta                338
California             338
Charlotte              338
Chicago                338
CincinnatiDayton       338
Columbus               338
DallasFtWorth          338
Denver                 338
Detroit                338
GrandRapids            338
GreatLakes             338
HarrisburgScranton     338
HartfordSpringfield    338
Houston                338
Indianapolis           338
Jacksonville           338
BuffaloRochester       338
LasVegas               338
LosAngeles             338
MiamiFtLauderdale      338
Louisv

1.2 Write Python code to check for duplicate rows and missing values in the dataset. Show the number of duplicates and missing values for each column. Then, explain (in comments or markdown) how you would handle these issues (e.g., drop, impute, or replace).

In [4]:
# Write your code here

import pandas as pd

duplicate_rows = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicate_rows}")


missing_values = df.isnull().sum()
print("\nNumber of missing values per column:")
print(missing_values[missing_values > 0])

total_missing_values = df.isnull().sum().sum()
print(f"\nTotal number of missing values in the dataset: {total_missing_values}")

# Personally I would handle duplicate rows using drop and missing values with impute.
# The reason for this is that with duplicate rows drop would simply remove the data,
# since it is a duplicate row it is likely not meant to be there and usually
# best for it to be removed. For missing values I would prefer impute since that
# would fill in values with estimations that take previous values into account,
# this is not perfect but likely the best way to ensure accurate data.


Number of duplicate rows: 2

Number of missing values per column:
Total Volume    1
4046            2
4225            1
4770            1
Total Bags      1
Small Bags      2
Large Bags      2
XLarge Bags     1
type            1
dtype: int64

Total number of missing values in the dataset: 12


1.3 Use Python code to print the number of rows and columns in the dataset (e.g., with df.shape). Based on the dataset size, explain (briefly) whether you think the dataset is sufficient for training a machine learning model.

In [5]:
# Write your code here

# Using df.shape to get the number of rows and columns
rows, cols = df.shape
print(f"The dataset has {rows} rows and {cols} columns.")

# I think overall this is a fine dataset for training a machine learning model.
# There is a good quantity of data, not a ton but a lot more than would be reasonable
# for a human to analyze. The quality is also relatively strong, it does have
# some errors but very few and far between and ones that could easily be fixed
# with a little extra cod

The dataset has 18254 rows and 14 columns.


#2. Data Cleaning and Preprocessing

2.1 Remove the first column or “Column 1” from the dataset. Treat the ‘year’ variable as nominal.

In [6]:
# Write your code here
df = df.iloc[:, 1:]

if 'year' in df.columns:
    df['year'] = df['year'].astype('category')

print(df.head())

         Date  AveragePrice  Total Volume     4046       4225    4770  \
0  12-27-2015          1.33      64236.62  1036.74   54454.85   48.16   
1  12-20-2015          1.35      54876.98   674.28   44638.81   58.33   
2  12-13-2015          0.93     118220.22   794.70  109149.67  130.50   
3   12-6-2015          1.08      78992.15  1132.00   71976.41   72.58   
4  11-29-2015          1.28      51039.60   941.48   43838.39   75.78   

   Total Bags  Small Bags  Large Bags  XLarge Bags          type  year  region  
0     8696.87     8603.62       93.25          0.0  conventional  2015  Albany  
1     9505.56     9408.07       97.49          0.0  conventional  2015  Albany  
2     8145.35     8042.21      103.14          0.0  conventional  2015  Albany  
3     5811.16     5677.40      133.76          0.0  conventional  2015  Albany  
4     6183.95     5986.26      197.69          0.0  conventional  2015  Albany  


2.2 Check for duplicate values and remove them.

In [8]:
# Write your code here
duplicate_rows = df.duplicated().sum()

df = df.drop_duplicates()

2.3 Check for missing values. If a data record (row) only has a few missing values, replace the missing values with the median of the column feature in that specific “Region” variable. If most column values in a data record are missing, remove the data record.

In [10]:
# Write your code here
num_columns = df.shape[1]

threshold_for_removing_row = num_columns / 2

missing_per_row = df.isnull().sum(axis=1)

rows_to_remove_indices = df[missing_per_row > threshold_for_removing_row].index

df_cleaned = df.drop(rows_to_remove_indices)


numeric_cols_with_missing = df_cleaned.select_dtypes(include=['number']).columns[df_cleaned.select_dtypes(include=['number']).isnull().any()].tolist()

non_numeric_cols_with_missing = df_cleaned.select_dtypes(exclude=['number']).columns[df_cleaned.select_dtypes(exclude=['number']).isnull().any()].tolist()

if 'region' in df_cleaned.columns:
    for col in numeric_cols_with_missing:
        df_cleaned[col] = df_cleaned.groupby('region')[col].transform(lambda x: x.fillna(x.median()))
else:
    for col in numeric_cols_with_missing:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

for col in non_numeric_cols_with_missing:
    if not df_cleaned[col].mode().empty:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])
    else:
        df_cleaned[col] = df_cleaned[col].fillna('Unknown')

df = df_cleaned.copy()

2.4 Find the correlation between the variables and describe how the correlated values among the variables impact the model accuracy.


In [39]:
# Write your code here
correlation_matrix = df.corr(numeric_only=True)

print(correlation_matrix)

# Alot of the data appears to have relatively strong correlation with the
# exception of average price having particularly low correlation and XLarge bags
# appearing to have somewhat lower correlation in comparison to every other
# column.

              AveragePrice  Total Volume      4046      4225      4770  \
AveragePrice      1.000000     -0.192767 -0.208325 -0.172944 -0.179458   
Total Volume     -0.192767      1.000000  0.977863  0.974181  0.872203   
4046             -0.208325      0.977863  1.000000  0.926110  0.833390   
4225             -0.172944      0.974181  0.926110  1.000000  0.887856   
4770             -0.179458      0.872203  0.833390  0.887856  1.000000   
Total Bags       -0.177103      0.963047  0.920057  0.905788  0.792315   
Small Bags       -0.174742      0.967238  0.925280  0.916032  0.802734   
Large Bags       -0.172953      0.880640  0.838646  0.810016  0.698473   
XLarge Bags      -0.117604      0.747158  0.699378  0.688810  0.679862   

              Total Bags  Small Bags  Large Bags  XLarge Bags  
AveragePrice   -0.177103   -0.174742   -0.172953    -0.117604  
Total Volume    0.963047    0.967238    0.880640     0.747158  
4046            0.920057    0.925280    0.838646     0.699378  
422

#3. Exploratory Data Analysis (EDA)


3.1 Describe the variables
- Describe all variables in the dataset.
- For continuous variables: report **range (min, max), mean, median, and distribution**.
- For categorical variables: list unique values.

In [17]:
import pandas as pd

for column in df.columns:
    print(f"--- Column: {column} ---")

    if pd.api.types.is_numeric_dtype(df[column]):
        print("Type: Continuous")
        print(f"Range: ({df[column].min()}, {df[column].max()})")
        print(f"Mean: {df[column].mean():.2f}")
        print(f"Median: {df[column].median():.2f}")
        print("Distribution: Skewness and kurtosis can be checked, or visualize with histogram/KDE.")
        print(df[column].describe())

--- Column: Date ---
--- Column: AveragePrice ---
Type: Continuous
Range: (0.44, 3.25)
Mean: 1.41
Median: 1.37
Distribution: Skewness and kurtosis can be checked, or visualize with histogram/KDE.
count    18251.000000
mean         1.406020
std          0.402675
min          0.440000
25%          1.100000
50%          1.370000
75%          1.660000
max          3.250000
Name: AveragePrice, dtype: float64
--- Column: Total Volume ---
Type: Continuous
Range: (84.56, 62505646.52)
Mean: 850552.31
Median: 107354.25
Distribution: Skewness and kurtosis can be checked, or visualize with histogram/KDE.
count    1.825100e+04
mean     8.505523e+05
std      3.453367e+06
min      8.456000e+01
25%      1.084067e+04
50%      1.073542e+05
75%      4.329430e+05
max      6.250565e+07
Name: Total Volume, dtype: float64
--- Column: 4046 ---
Type: Continuous
Range: (0.0, 22743616.17)
Mean: 292983.95
Median: 8645.30
Distribution: Skewness and kurtosis can be checked, or visualize with histogram/KDE.
count   

3.2 Inspect the earliest recorded date
- Find the earliest `Date`.
- Check if there are avocado prices recorded from the earliest date up to 2010.
- Comment: does the earliest data point look reasonable? Keep or remove?

In [37]:
# Write your code here
df['Date'] = pd.to_datetime(df['Date'])

earliest_date = df['Date'].min()
print(f"Earliest recorded date: {earliest_date.strftime('%Y-%m-%d')}")

data_up_to_2010 = df[df['Date'].dt.year <= 2010]

print(f"Amount of recorded avocado prices from {earliest_date.year} to 2010: {len(data_up_to_2010)}")

# Considering the earliest data point is all the way back in 1904 and it is the
# only record of avocado prices from then to 2010 it is most reasonable to
# remove the data point entirely.

Earliest recorded date: 1904-01-21
Amount of recorded avocado prices from 1904 to 2010: 1


3.3 Highest average price
- Find the highest value in "AveragePrice".
- Report which region it belongs to.
- Describe how you obtained the result.

In [38]:
# Write your code here
highest_value = df['AveragePrice'].max()
print(f"Highest value in 'AveragePrice': {highest_value}")

highest_value_region = df[df['AveragePrice'] == highest_value]['region'].iloc[0]
print(f"Region with highest value in 'AveragePrice': {highest_value_region}")

# I obtained the result by creating a variable 'highest_value' and setting it
# to the max value within the 'AveragePrice' column. Next I created a
# 'highest_value_region' variable and set it to the variable in the 'region'
# column that corresponds to the 'highest_value' variable.

Highest value in 'AveragePrice': 3.25
Region with highest value in 'AveragePrice': SanFrancisco


3.4 Highest total volume
- Find the highest total volume of avocados.
- Report which region it belongs to.
- Describe how you obtained the result.

In [36]:
# Write your code here
highest_total_volume = df['Total Volume'].max()
print(f"Highest total volume of avocados: {highest_total_volume}")

highest_total_volume_region = df[df['Total Volume'] == highest_total_volume]['region'].iloc[0]
print(f"Region with highest total volume of avocados: {highest_total_volume_region}")

# Very similar to 3.3, I created a variable and set it to the maximum value in
# the 'Total Volume' column. Next I created a variable and set it to the variable
# in the 'region' column that corresponds to the 'highest_total_volume'.

Highest total volume of avocados: 62505646.52
Region with highest total volume of avocados: TotalUS
